# Soccer Video Analysis (RunPod + Colab)

This notebook runs full soccer video analysis using the sports project.

Steps included:
- Detect local repo and install dependencies (offline friendly)
- Ensure pretrained models are available
- Use local video path (RunPod) or optional upload (Colab)
- Run analysis in selected mode
- Generate advanced stats and charts
- Export stats ZIP and result video

In [34]:
# Runtime check
!nvidia-smi

Thu Mar 12 23:38:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P0             30W /   70W |    3193MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Locate repository and install dependencies (RunPod/Colab/offline friendly)
import os
import sys
import socket
import subprocess
from pathlib import Path

def _run(cmd, cwd=None, check=True):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, check=check)

def _has_internet(host="github.com", port=443, timeout=3):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except Exception:
        return False

candidates = [
    "/workspace/sports_repo",
    "/workspace/sports",
    "/content/sports_repo",
    "/content/sports",
    str(Path.cwd()),
]

REPO_ROOT = None
for candidate in candidates:
    if os.path.exists(os.path.join(candidate, "setup.py")) and os.path.exists(
        os.path.join(candidate, "examples", "soccer")
    ):
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    if os.path.exists("/workspace"):
        parent = "/workspace"
    elif os.path.exists("/content"):
        parent = "/content"
    else:
        parent = str(Path.cwd())

    clone_target = os.path.join(parent, "sports_repo")
    if not os.path.exists(clone_target):
        if _has_internet():
            _run(["git", "clone", "https://github.com/roboflow/sports.git", clone_target], check=True)
        else:
            raise RuntimeError(
                "Sports repo not found locally and internet is unavailable. "
                "Place the repo at /workspace/sports_repo or /workspace/sports."
            )
    REPO_ROOT = clone_target

SOCCER_DIR = os.path.join(REPO_ROOT, "examples", "soccer")
DATA_DIR = os.path.join(SOCCER_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)

pip_cmd = [sys.executable, "-m", "pip"]
_run(pip_cmd + ["install", "-q", "-U", "pip"], check=False)

# Always install local package from disk (works offline).
_run(pip_cmd + ["install", "-q", "-e", ".", "--no-deps"], check=True)

req_path = os.path.join(REPO_ROOT, "examples", "soccer", "requirements.txt")
if os.path.exists(req_path):
    try:
        _run(pip_cmd + ["install", "-q", "-r", req_path], check=True)
    except Exception as ex:
        print("requirements install failed/skipped:", ex)
        print("Continuing with already-installed environment packages.")
else:
    print("requirements.txt not found, skipping extra dependency install.")

/content
fatal: destination path 'sports_repo' already exists and is not an empty directory.
/content/sports_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sports (pyproject.toml) ... done


In [ ]:
# Ensure model assets exist (download when internet is available)
import os
import subprocess

SOCCER_DIR = os.path.join(REPO_ROOT, "examples", "soccer")
DATA_DIR = os.path.join(SOCCER_DIR, "data")
os.makedirs(DATA_DIR, exist_ok=True)

required_models = [
    "football-ball-detection.pt",
    "football-player-detection.pt",
    "football-pitch-detection.pt",
]
missing = [name for name in required_models if not os.path.exists(os.path.join(DATA_DIR, name))]

if not missing:
    print("All required model files already exist in", DATA_DIR)
else:
    print("Missing model files:", missing)
    setup_script = os.path.join(SOCCER_DIR, "setup.sh")
    try:
        subprocess.run(["bash", setup_script], check=True)
    except Exception:
        print("Auto-download failed (likely offline).")
        print("Please place these files manually in:", DATA_DIR)
        print("-", "\n- ".join(required_models))

missing_after = [name for name in required_models if not os.path.exists(os.path.join(DATA_DIR, name))]
if missing_after:
    raise FileNotFoundError(
        "Missing required model files after setup: " + ", ".join(missing_after)
    )
print("Model assets ready.")

/content/sports_repo/examples/soccer
'data' directory already exists.
Downloading...
From (original): https://drive.google.com/uc?id=1isw4wx-MK9h9LMr36VvIWlJD6ppUvw7V
From (redirected): https://drive.google.com/uc?id=1isw4wx-MK9h9LMr36VvIWlJD6ppUvw7V&confirm=t&uuid=43b69297-6a0a-48d6-8ad8-734b146b0b0d
To: /content/sports_repo/examples/soccer/data/football-ball-detection.pt
100% 137M/137M [00:05<00:00, 26.5MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=17PXFNlx-jI7VjVo_vQnB1sONjRyvoB-q
From (redirected): https://drive.google.com/uc?id=17PXFNlx-jI7VjVo_vQnB1sONjRyvoB-q&confirm=t&uuid=1b980f87-af5a-400a-a859-119ee98bb509
To: /content/sports_repo/examples/soccer/data/football-player-detection.pt
100% 137M/137M [00:03<00:00, 45.3MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Ma5Kt86tgpdjCTKfum79YMgNnSjcoOyf
From (redirected): https://drive.google.com/uc?id=1Ma5Kt86tgpdjCTKfum79YMgNnSjcoOyf&confirm=t&uuid=8d4bc65e-9456-4558-bad9-5e3945dc3ddd
To: /

In [ ]:
# Configuration + choose mode + select video (RunPod/Colab)
MODE = "PLAYER_DETECTION" #@param ["PITCH_DETECTION", "PLAYER_DETECTION", "BALL_DETECTION", "PLAYER_TRACKING", "TEAM_CLASSIFICATION", "RADAR"]

# Team orientation setup
MY_TEAM_ID = 0                 # 0 or 1 (cluster ID of your team)
MY_TEAM_OWN_GOAL_SIDE = "left"  # "left" or "right"

# For RunPod/local Jupyter: set full path to your video
SOURCE_VIDEO_PATH_INPUT = "/workspace/input.mp4"  # change this path

# For Colab only: set True to upload from browser
USE_COLAB_UPLOAD = False

from pathlib import Path
import os
import subprocess
import cv2
import torch

DATA_DIR = os.path.join(REPO_ROOT, "examples", "soccer", "data")
os.makedirs(DATA_DIR, exist_ok=True)

def _can_open_video(path: str) -> bool:
    cap = cv2.VideoCapture(path)
    ok = cap.isOpened()
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) if ok else 0
    cap.release()
    return ok and frame_count > 0

# Resolve input source
SOURCE_VIDEO_PATH = None
uploaded_name = None

try:
    from google.colab import files as colab_files  # type: ignore
    _in_colab = True
except Exception:
    _in_colab = False

if _in_colab and USE_COLAB_UPLOAD:
    print("Please upload your video file:")
    uploaded = colab_files.upload()
    if not uploaded:
        raise ValueError("No file uploaded.")
    uploaded_name = next(iter(uploaded.keys()))
    tmp_path = str(Path("/content") / uploaded_name)
    with open(tmp_path, "wb") as f:
        f.write(uploaded[uploaded_name])
    SOURCE_VIDEO_PATH = tmp_path
else:
    SOURCE_VIDEO_PATH = SOURCE_VIDEO_PATH_INPUT
    if not os.path.exists(SOURCE_VIDEO_PATH):
        raise FileNotFoundError(
            f"Input video not found: {SOURCE_VIDEO_PATH}. Set SOURCE_VIDEO_PATH_INPUT correctly."
        )
    uploaded_name = os.path.basename(SOURCE_VIDEO_PATH)

# Normalize video codec if needed
if not _can_open_video(SOURCE_VIDEO_PATH):
    fixed_video_path = str(Path(DATA_DIR) / f"{Path(uploaded_name).stem}_h264.mp4")
    print("Video not readable by OpenCV. Re-encoding to H.264/AAC...")
    ffmpeg_cmd = [
        "ffmpeg", "-y",
        "-i", SOURCE_VIDEO_PATH,
        "-c:v", "libx264",
        "-preset", "fast",
        "-crf", "23",
        "-c:a", "aac",
        "-movflags", "+faststart",
        fixed_video_path,
    ]
    result = subprocess.run(ffmpeg_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if result.returncode != 0 or not _can_open_video(fixed_video_path):
        print(result.stdout[-3000:])
        raise ValueError("Video conversion failed. Use a standard mp4 (H.264/AAC).")
    SOURCE_VIDEO_PATH = fixed_video_path
    print("Using converted video:", SOURCE_VIDEO_PATH)

TARGET_VIDEO_PATH = os.path.join(
    DATA_DIR,
    f"output-{MODE.lower()}_{Path(uploaded_name).stem}.mp4",
)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Mode       :", MODE)
print("My team ID :", MY_TEAM_ID)
print("My goal    :", MY_TEAM_OWN_GOAL_SIDE)
print("Device     :", DEVICE)
print("Source     :", SOURCE_VIDEO_PATH)
print("Target     :", TARGET_VIDEO_PATH)

Please upload your video file:


Saving 3.mp4 to 3.mp4
Mode       : PLAYER_DETECTION
My team ID : 0
My goal    : left
Device     : cuda
Source     : /content/sports_repo/examples/soccer/data/3.mp4
Target     : /content/sports_repo/examples/soccer/data/output-player_detection_3.mp4
Uploaded file saved to: /content/sports_repo/examples/soccer/data/3.mp4


In [39]:
# Run analysis without GUI calls
import sys, os
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(f"{REPO_ROOT}/examples/soccer")
import supervision as sv
import main as soccer_main
from tqdm.notebook import tqdm # Import tqdm for progress bar

def run_colab(source_video_path, target_video_path, device, mode):
    mode_map = {
        "PITCH_DETECTION": soccer_main.run_pitch_detection,
        "PLAYER_DETECTION": soccer_main.run_player_detection,
        "BALL_DETECTION": soccer_main.run_ball_detection,
        "PLAYER_TRACKING": soccer_main.run_player_tracking,
        "TEAM_CLASSIFICATION": soccer_main.run_team_classification,
        "RADAR": soccer_main.run_radar,
    }

    if mode not in mode_map:
        raise ValueError(f"Unsupported mode: {mode}")

    # Diagnostic check - keep as a safeguard
    if not os.path.exists(source_video_path):
        raise FileNotFoundError(f"Video file not found at: {source_video_path}. Please ensure the file was uploaded correctly in the previous step.")

    frame_generator = mode_map[mode](source_video_path=source_video_path, device=device)
    video_info = sv.VideoInfo.from_video_path(source_video_path)

    print(f"Starting analysis in {mode} mode for {os.path.basename(source_video_path)}...")

    with sv.VideoSink(target_video_path, video_info) as sink:
        # Wrap frame_generator with tqdm for a progress bar
        for frame in tqdm(frame_generator, total=video_info.total_frames, desc=f"Processing video ({mode})"):
            sink.write_frame(frame)

    print("Saved result to:", target_video_path)

run_colab(SOURCE_VIDEO_PATH, TARGET_VIDEO_PATH, DEVICE, MODE)

Starting analysis in PLAYER_DETECTION mode for 3.mp4...


Processing video (PLAYER_DETECTION):   0%|          | 0/2047 [00:00<?, ?it/s]

Saved result to: /content/sports_repo/examples/soccer/data/output-player_detection_3.mp4


In [ ]:
# Advanced match statistics (estimated from detections/tracking)
import os
import json
import csv
from collections import Counter, defaultdict
import numpy as np
import supervision as sv
from tqdm import tqdm
from ultralytics import YOLO
from sports.common.team import TeamClassifier, PlayerReIdentifier
from sports.common.ball import BallTracker
from sports.common.stats import PassDetector, PossessionTracker, TeamVoteBuffer

# --- Thresholds ---
MIN_OWNER_DIST_PX = 80
TEN_MINUTES_SEC = 600
TEAM_FIT_STRIDE = 60

# Pass quality filters (notes 2, 11, 12, 13)
MIN_BALL_SPEED_PX_PER_SEC = 300.0   # filter dribble noise
PASS_DEBOUNCE_SEC = 1.35            # ignore same-team passes within this window
PASS_MIN_DIST_PX = 85.0             # ignore short passes
ID_SWITCH_GUARD_PX = 65.0           # same-team change < this distance = tracker ID switch

# Team temporal-confirmation buffer (notes 0, 8)
TEAM_VOTE_BUFFER_SIZE = 64
TEAM_VOTE_MIN_FRAMES = 8

# PlayerReIdentifier settings (notes 3, 4)
REID_POSITION_TOLERANCE_PX = 140.0
REID_MAX_FRAMES_LOST = 90

if MY_TEAM_ID not in (0, 1):
    raise ValueError("MY_TEAM_ID must be 0 or 1.")

my_goal_side = str(MY_TEAM_OWN_GOAL_SIDE).strip().lower()
if my_goal_side not in ("left", "right"):
    raise ValueError("MY_TEAM_OWN_GOAL_SIDE must be 'left' or 'right'.")

opponent_team_id = 1 - MY_TEAM_ID
team_goal_side = {
    MY_TEAM_ID: my_goal_side,
    opponent_team_id: ("right" if my_goal_side == "left" else "left"),
}

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
fps = float(video_info.fps) if video_info.fps else 25.0
total_frames = int(video_info.total_frames)
duration_sec = total_frames / fps if fps > 0 else 0.0

stats_dir = os.path.join(REPO_ROOT, "examples", "soccer", "data", "stats")
os.makedirs(stats_dir, exist_ok=True)

player_model = YOLO(soccer_main.PLAYER_DETECTION_MODEL_PATH).to(device=DEVICE)
ball_model = YOLO(soccer_main.BALL_DETECTION_MODEL_PATH).to(device=DEVICE)

# 1) Build team classifier from sampled player crops
fit_crops = []
fit_generator = sv.get_video_frames_generator(source_path=SOURCE_VIDEO_PATH, stride=TEAM_FIT_STRIDE)
for frame in tqdm(fit_generator, desc="collecting crops for team classifier"):
    player_result = player_model(frame, imgsz=1280, verbose=False)[0]
    det = sv.Detections.from_ultralytics(player_result)
    players_only = det[det.class_id == soccer_main.PLAYER_CLASS_ID]
    fit_crops += soccer_main.get_crops(frame, players_only)

if len(fit_crops) < 10:
    raise ValueError("Not enough player crops to fit team classifier. Try a clearer/longer video.")

team_classifier = TeamClassifier(device=DEVICE)
team_classifier.fit(fit_crops)

# 2) Prepare trackers, ReID and helpers
player_tracker = sv.ByteTrack(minimum_consecutive_frames=3)
ball_tracker = BallTracker(buffer_size=20)

# PlayerReIdentifier: maps short-lived ByteTrack IDs → stable canonical IDs (notes 3, 4)
reid = PlayerReIdentifier(
    max_frames_lost=REID_MAX_FRAMES_LOST,
    position_tolerance_px=REID_POSITION_TOLERANCE_PX,
)

# TeamVoteBuffer: temporal confirmation of team assignments per canonical ID (notes 0, 8)
team_vote_buf = TeamVoteBuffer(
    buffer_size=TEAM_VOTE_BUFFER_SIZE,
    min_votes=TEAM_VOTE_MIN_FRAMES,
)

# PassDetector: applies all pass quality filters (notes 2, 11, 12, 13)
pass_detector = PassDetector(
    min_ball_speed_px_per_sec=MIN_BALL_SPEED_PX_PER_SEC,
    pass_debounce_sec=PASS_DEBOUNCE_SEC,
    pass_min_dist_px=PASS_MIN_DIST_PX,
    id_switch_guard_px=ID_SWITCH_GUARD_PX,
    fps=fps,
)

# PossessionTracker: proximity-weighted frame counts (note 1)
possession_tracker = PossessionTracker(max_owner_dist_px=MIN_OWNER_DIST_PX)

def _ball_callback(image_slice: np.ndarray) -> sv.Detections:
    result = ball_model(image_slice, imgsz=640, verbose=False)[0]
    return sv.Detections.from_ultralytics(result)

ball_slicer = sv.InferenceSlicer(
    callback=_ball_callback,
    slice_wh=(640, 640),
)

def _to_track_id(value):
    if value is None:
        return None
    try:
        if np.isnan(value):
            return None
    except Exception:
        pass
    try:
        return int(value)
    except Exception:
        return None

def _safe_goalkeeper_team(players, players_team_id, goalkeepers):
    if len(goalkeepers) == 0:
        return np.array([], dtype=int)
    if len(players) == 0 or len(players_team_id) == 0:
        return np.full(len(goalkeepers), -1, dtype=int)
    valid_ids = players_team_id[players_team_id >= 0]
    unique = np.unique(valid_ids)
    if len(unique) == 0:
        return np.full(len(goalkeepers), -1, dtype=int)
    if len(unique) < 2:
        return np.full(len(goalkeepers), int(unique[0]), dtype=int)
    return soccer_main.resolve_goalkeepers_team_id(players, players_team_id, goalkeepers).astype(int)

def _base_third_from_x(x, frame_width):
    if x < frame_width / 3.0:
        return "left"
    if x < 2.0 * frame_width / 3.0:
        return "middle"
    return "right"

def _phase_third_for_team(base_third, team_id):
    if base_third == "middle":
        return "middle"
    side = team_goal_side.get(team_id, None)
    if side is None:
        return None
    if side == "left":
        return "defensive" if base_third == "left" else "offensive"
    return "defensive" if base_third == "right" else "offensive"

# 3) Stats containers
window_total_frames = Counter()
window_team_frames = defaultdict(Counter)
team_touches = Counter()
team_passes_completed = Counter()
team_turnovers_lost = Counter()
team_interceptions_won = Counter()
team_detected_count_sum = Counter()

# Third-based possession (weighted)
team_third_possession_frames = defaultdict(Counter)
window_team_third_frames = defaultdict(lambda: defaultdict(Counter))

player_touches = Counter()
player_possession_frames = Counter()   # weighted
player_distance_px = Counter()
last_player_xy = {}

# Canonical-ID → team vote accumulator (for final aggregation)
tracker_team_votes = defaultdict(Counter)

events = []
last_ball_xy = None
ball_distance_px = 0.0

current_owner = None            # (canonical_id, team_id)
current_owner_start_frame = 0
current_owner_ball_xy = None    # ball position when ownership started

min_touch_frames = max(2, int(0.15 * fps))

frame_generator = sv.get_video_frames_generator(source_path=SOURCE_VIDEO_PATH)
for frame_idx, frame in enumerate(tqdm(frame_generator, total=total_frames, desc="extracting stats")):
    # --- Player detection + tracking ---
    player_result = player_model(frame, imgsz=1280, verbose=False)[0]
    detections = sv.Detections.from_ultralytics(player_result)
    detections = player_tracker.update_with_detections(detections)

    players = detections[detections.class_id == soccer_main.PLAYER_CLASS_ID]
    goalkeepers = detections[detections.class_id == soccer_main.GOALKEEPER_CLASS_ID]

    # --- Team classification with temporal stabilisation (notes 0, 3, 4, 8) ---
    player_crops = soccer_main.get_crops(frame, players)
    players_team_id_raw = (
        team_classifier.predict(player_crops).astype(int)
        if len(players) > 0
        else np.array([], dtype=int)
    )

    # Resolve stable canonical IDs and temporally-confirmed team IDs
    players_team_id = players_team_id_raw.copy() if len(players_team_id_raw) > 0 else np.array([], dtype=int)
    player_canonical_ids = []

    if len(players) > 0:
        players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        players_tid_arr = players.tracker_id if players.tracker_id is not None else [None] * len(players)

        for i in range(len(players)):
            tid = _to_track_id(players_tid_arr[i])
            xy = players_xy[i]
            raw_team = int(players_team_id_raw[i]) if i < len(players_team_id_raw) else -1

            if tid is not None:
                # Map ByteTrack ID → stable canonical ID (note 3, 4)
                canonical = reid.get_stable_id(
                    tid, xy, team_id=raw_team if raw_team in (0, 1) else None
                )
                # Temporal team confirmation via vote buffer (notes 0, 8)
                stable_team = team_vote_buf.update(canonical, raw_team)
                players_team_id[i] = stable_team

                if stable_team in (0, 1):
                    tracker_team_votes[canonical][stable_team] += 1

                # Track player movement
                prev_xy = last_player_xy.get(canonical)
                if prev_xy is not None:
                    player_distance_px[canonical] += float(np.linalg.norm(xy - prev_xy))
                last_player_xy[canonical] = xy

                player_canonical_ids.append(canonical)
            else:
                player_canonical_ids.append(None)

    goalkeepers_team_id = _safe_goalkeeper_team(players, players_team_id, goalkeepers)

    # --- Build owner-candidate list ---
    owner_candidates = []

    if len(players) > 0:
        players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        for i in range(len(players)):
            canonical = player_canonical_ids[i] if i < len(player_canonical_ids) else None
            team_id = int(players_team_id[i]) if i < len(players_team_id) else -1
            if team_id in (0, 1):
                team_detected_count_sum[team_id] += 1
            owner_candidates.append({"canonical": canonical, "team": team_id, "xy": players_xy[i]})

    if len(goalkeepers) > 0:
        gk_xy = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        gk_tid_arr = goalkeepers.tracker_id if goalkeepers.tracker_id is not None else [None] * len(goalkeepers)
        for i in range(len(goalkeepers)):
            tid = _to_track_id(gk_tid_arr[i])
            team_id = int(goalkeepers_team_id[i]) if i < len(goalkeepers_team_id) else -1
            xy = gk_xy[i]
            canonical = None
            if tid is not None:
                canonical = reid.get_stable_id(tid, xy, team_id=team_id if team_id in (0, 1) else None)
                prev_xy = last_player_xy.get(canonical)
                if prev_xy is not None:
                    player_distance_px[canonical] += float(np.linalg.norm(xy - prev_xy))
                last_player_xy[canonical] = xy
                if team_id in (0, 1):
                    tracker_team_votes[canonical][team_id] += 1
            owner_candidates.append({"canonical": canonical, "team": team_id, "xy": xy})

    # Age the ReID gallery (note 3)
    reid.end_frame()

    # --- Ball detection ---
    ball_det = ball_slicer(frame).with_nms(threshold=0.1)
    ball_det = ball_tracker.update(ball_det)

    ball_xy = None
    ball_speed = 0.0
    if len(ball_det) > 0:
        ball_xy = ball_det.get_anchors_coordinates(sv.Position.CENTER)[0]
        if last_ball_xy is not None:
            dist = float(np.linalg.norm(ball_xy - last_ball_xy))
            ball_distance_px += dist
            ball_speed = dist * fps  # px/sec
        last_ball_xy = ball_xy

    # --- Possession: nearest player to ball with proximity weighting (note 1) ---
    owner_candidate = None
    owner_weight = 0.0
    if ball_xy is not None and len(owner_candidates) > 0:
        distances = [float(np.linalg.norm(c["xy"] - ball_xy)) for c in owner_candidates]
        nearest_idx = int(np.argmin(distances))
        nearest_dist = distances[nearest_idx]
        if nearest_dist <= MIN_OWNER_DIST_PX:
            nearest = owner_candidates[nearest_idx]
            if nearest["team"] in (0, 1):
                owner_candidate = (nearest["canonical"], nearest["team"])
                owner_weight = possession_tracker.update(
                    nearest["team"], ball_xy, nearest["xy"]
                )

    # --- Possession aggregation ---
    window_idx = int((frame_idx / fps) // TEN_MINUTES_SEC) if fps > 0 else 0
    window_total_frames[window_idx] += 1

    if owner_candidate is not None:
        owner_cid, owner_team = owner_candidate
        window_team_frames[window_idx][owner_team] += owner_weight
        if owner_cid is not None:
            player_possession_frames[owner_cid] += owner_weight

        if ball_xy is not None:
            base_third = _base_third_from_x(float(ball_xy[0]), frame.shape[1])
            phase_third = _phase_third_for_team(base_third, owner_team)
            if phase_third is not None:
                team_third_possession_frames[owner_team][phase_third] += owner_weight
                window_team_third_frames[window_idx][owner_team][phase_third] += owner_weight

    # --- Touches / passes / turnovers ---
    if owner_candidate != current_owner:
        if owner_candidate is not None:
            new_cid, new_team = owner_candidate
            team_touches[new_team] += 1
            if new_cid is not None:
                player_touches[new_cid] += 1

        prev_duration = frame_idx - current_owner_start_frame
        if current_owner is not None and owner_candidate is not None and prev_duration >= min_touch_frames:
            prev_cid, prev_team = current_owner
            new_cid, new_team = owner_candidate
            ts = frame_idx / fps if fps > 0 else 0.0

            # Ball displacement since previous owner received the ball
            ball_displacement = 0.0
            if current_owner_ball_xy is not None and ball_xy is not None:
                ball_displacement = float(np.linalg.norm(
                    np.asarray(ball_xy) - np.asarray(current_owner_ball_xy)
                ))

            prev_pos = last_player_xy.get(prev_cid) if prev_cid is not None else None
            new_pos = last_player_xy.get(new_cid) if new_cid is not None else None

            if prev_team == new_team:
                # Same-team transfer: apply pass quality filters (notes 2, 11, 12, 13, 15)
                is_pass = pass_detector.check(
                    prev_canonical_id=prev_cid,
                    prev_team_id=prev_team,
                    new_canonical_id=new_cid,
                    new_team_id=new_team,
                    prev_player_xy=prev_pos,
                    new_player_xy=new_pos,
                    ball_speed_px_per_sec=ball_speed,
                    ball_displacement_px=ball_displacement,
                    time_sec=ts,
                )
                if is_pass:
                    team_passes_completed[prev_team] += 1
                    events.append({
                        "time_sec": round(ts, 2),
                        "event": "pass_completed",
                        "team": int(prev_team),
                        "from_canonical": (None if prev_cid is None else int(prev_cid)),
                        "to_canonical": (None if new_cid is None else int(new_cid)),
                    })
            else:
                # Cross-team transfer: turnover / interception (note 10)
                team_turnovers_lost[prev_team] += 1
                team_interceptions_won[new_team] += 1
                events.append({
                    "time_sec": round(ts, 2),
                    "event": "turnover_interception",
                    "lost_by_team": int(prev_team),
                    "won_by_team": int(new_team),
                    "from_canonical": (None if prev_cid is None else int(prev_cid)),
                    "to_canonical": (None if new_cid is None else int(new_cid)),
                })

        current_owner = owner_candidate
        current_owner_start_frame = frame_idx
        current_owner_ball_xy = ball_xy

# 4) Final aggregation
tracker_team_final = {}
for cid, votes in tracker_team_votes.items():
    if len(votes) > 0:
        tracker_team_final[cid] = votes.most_common(1)[0][0]

team_distance_px = Counter()
for cid, dist in player_distance_px.items():
    team_id = tracker_team_final.get(cid, None)
    if team_id in (0, 1):
        team_distance_px[team_id] += float(dist)

def _pct(n, d):
    return 100.0 * float(n) / float(d) if d else 0.0

team_0_frames = possession_tracker.weighted_frames(0)
team_1_frames = possession_tracker.weighted_frames(1)
unknown_frames = max(total_frames - team_0_frames - team_1_frames, 0)
known_frames = team_0_frames + team_1_frames

team_stats_rows = []
for team_id in [0, 1]:
    passes_completed = team_passes_completed[team_id]
    pass_attempts_est = passes_completed + team_turnovers_lost[team_id]
    poss_frames = possession_tracker.weighted_frames(team_id)
    team_stats_rows.append({
        "team_id": team_id,
        "possession_pct_total_frames": round(_pct(poss_frames, total_frames), 2),
        "possession_pct_when_known": round(_pct(poss_frames, known_frames), 2),
        "touches": int(team_touches[team_id]),
        "passes_completed_est": int(passes_completed),
        "pass_attempts_est": int(pass_attempts_est),
        "pass_accuracy_est_pct": round(_pct(passes_completed, pass_attempts_est), 2),
        "turnovers_lost": int(team_turnovers_lost[team_id]),
        "interceptions_won": int(team_interceptions_won[team_id]),
        "distance_covered_px": round(float(team_distance_px[team_id]), 2),
        "avg_detected_players_per_frame": round(
            float(team_detected_count_sum[team_id]) / float(total_frames if total_frames else 1), 2
        ),
    })

window_rows = []
for w in sorted(window_total_frames.keys()):
    w_total = window_total_frames[w]
    w_t0 = window_team_frames[w][0]
    w_t1 = window_team_frames[w][1]
    w_unknown = max(w_total - w_t0 - w_t1, 0)
    start_sec = int(w * TEN_MINUTES_SEC)
    end_sec = int(min(duration_sec, (w + 1) * TEN_MINUTES_SEC))
    window_rows.append({
        "window_index": int(w),
        "start_sec": start_sec,
        "end_sec": end_sec,
        "team_0_possession_pct": round(_pct(w_t0, w_total), 2),
        "team_1_possession_pct": round(_pct(w_t1, w_total), 2),
        "unknown_pct": round(_pct(w_unknown, w_total), 2),
        "frames": int(w_total),
    })

# Third-based rows
third_labels = ["defensive", "middle", "offensive"]
team_thirds_rows = []
for team_id in [0, 1]:
    team_frames = sum(team_third_possession_frames[team_id][k] for k in third_labels)
    row = {
        "team_id": int(team_id),
        "goal_side": team_goal_side.get(team_id, "unknown"),
        "defensive_frames": round(team_third_possession_frames[team_id]["defensive"], 4),
        "middle_frames": round(team_third_possession_frames[team_id]["middle"], 4),
        "offensive_frames": round(team_third_possession_frames[team_id]["offensive"], 4),
        "defensive_sec": round(team_third_possession_frames[team_id]["defensive"] / fps if fps > 0 else 0.0, 2),
        "middle_sec": round(team_third_possession_frames[team_id]["middle"] / fps if fps > 0 else 0.0, 2),
        "offensive_sec": round(team_third_possession_frames[team_id]["offensive"] / fps if fps > 0 else 0.0, 2),
        "defensive_pct_of_team_possession": round(_pct(team_third_possession_frames[team_id]["defensive"], team_frames), 2),
        "middle_pct_of_team_possession": round(_pct(team_third_possession_frames[team_id]["middle"], team_frames), 2),
        "offensive_pct_of_team_possession": round(_pct(team_third_possession_frames[team_id]["offensive"], team_frames), 2),
        "defensive_pct_of_total_frames": round(_pct(team_third_possession_frames[team_id]["defensive"], total_frames), 2),
        "middle_pct_of_total_frames": round(_pct(team_third_possession_frames[team_id]["middle"], total_frames), 2),
        "offensive_pct_of_total_frames": round(_pct(team_third_possession_frames[team_id]["offensive"], total_frames), 2),
    }
    team_thirds_rows.append(row)

window_thirds_rows = []
for w in sorted(window_total_frames.keys()):
    start_sec = int(w * TEN_MINUTES_SEC)
    end_sec = int(min(duration_sec, (w + 1) * TEN_MINUTES_SEC))
    for team_id in [0, 1]:
        def_f = window_team_third_frames[w][team_id]["defensive"]
        mid_f = window_team_third_frames[w][team_id]["middle"]
        off_f = window_team_third_frames[w][team_id]["offensive"]
        team_w_frames = def_f + mid_f + off_f
        window_thirds_rows.append({
            "window_index": int(w),
            "start_sec": start_sec,
            "end_sec": end_sec,
            "team_id": int(team_id),
            "goal_side": team_goal_side.get(team_id, "unknown"),
            "defensive_frames": round(def_f, 4),
            "middle_frames": round(mid_f, 4),
            "offensive_frames": round(off_f, 4),
            "defensive_sec": round(def_f / fps if fps > 0 else 0.0, 2),
            "middle_sec": round(mid_f / fps if fps > 0 else 0.0, 2),
            "offensive_sec": round(off_f / fps if fps > 0 else 0.0, 2),
            "defensive_pct_of_team_window": round(_pct(def_f, team_w_frames), 2),
            "middle_pct_of_team_window": round(_pct(mid_f, team_w_frames), 2),
            "offensive_pct_of_team_window": round(_pct(off_f, team_w_frames), 2),
        })

player_rows = []
for cid in sorted(
    set(player_touches.keys()) | set(player_possession_frames.keys()) | set(player_distance_px.keys())
):
    team_id = tracker_team_final.get(cid, -1)
    player_rows.append({
        "canonical_id": int(cid),
        "team_id": int(team_id),
        "touches": int(player_touches[cid]),
        "possession_seconds": round(float(player_possession_frames[cid]) / fps if fps > 0 else 0.0, 2),
        "distance_covered_px": round(float(player_distance_px[cid]), 2),
    })

my_team_thirds = {
    "team_id": int(MY_TEAM_ID),
    "goal_side": team_goal_side[MY_TEAM_ID],
    "defensive_sec": round(team_third_possession_frames[MY_TEAM_ID]["defensive"] / fps if fps > 0 else 0.0, 2),
    "middle_sec": round(team_third_possession_frames[MY_TEAM_ID]["middle"] / fps if fps > 0 else 0.0, 2),
    "offensive_sec": round(team_third_possession_frames[MY_TEAM_ID]["offensive"] / fps if fps > 0 else 0.0, 2),
}

summary = {
    "source_video_path": SOURCE_VIDEO_PATH,
    "fps": round(fps, 3),
    "frame_step": 1,
    "total_frames_original": int(total_frames),
    "total_frames_estimated": int(total_frames),
    "duration_sec": round(duration_sec, 2),
    "duration_min": round(duration_sec / 60.0, 2),
    "ball_distance_px": round(ball_distance_px, 2),
    "ball_avg_speed_px_per_sec": round((ball_distance_px / duration_sec) if duration_sec > 0 else 0.0, 2),
    "my_team": {
        "team_id": int(MY_TEAM_ID),
        "own_goal_side": my_goal_side,
    },
    "possession": {
        "team_0_pct_total_frames": round(_pct(team_0_frames, total_frames), 2),
        "team_1_pct_total_frames": round(_pct(team_1_frames, total_frames), 2),
        "unknown_pct_total_frames": round(_pct(unknown_frames, total_frames), 2),
        "team_0_pct_when_known": round(_pct(team_0_frames, known_frames), 2),
        "team_1_pct_when_known": round(_pct(team_1_frames, known_frames), 2),
    },
    "thirds_possession": {
        "team_0": team_thirds_rows[0],
        "team_1": team_thirds_rows[1],
        "my_team": my_team_thirds,
    },
    "notes": [
        "Temporal confirmation stabilizes possession labels.",
        "Possession frames are weighted by ball-to-player proximity (0.5-1.0 scale).",
        "Pass events require ball speed >= MIN_BALL_SPEED_PX_PER_SEC to filter out dribble noise.",
        "PlayerReIdentifier re-maps short-lived ByteTrack IDs to stable canonical IDs.",
        f"ReID: position_tolerance={REID_POSITION_TOLERANCE_PX}px, max_frames_lost={REID_MAX_FRAMES_LOST}.",
        "Possession accuracy: reported both as % of total frames and % of known-possession frames.",
        "Batched player inference increases GPU utilization.",
        "Track cache reduces repeated embedding extraction.",
        f"Team buffer size={TEAM_VOTE_BUFFER_SIZE} batches unknown player classifications.",
        "Passing/interception stats use team-level ball displacement, not tracker-ID handoff.",
        "Release-to-reception path now counts only same-team passes, not opponent receptions.",
        f"ID switch guard: same-team owner changes under {ID_SWITCH_GUARD_PX}px are treated as the same player.",
        f"Pass debounce: ignore same-team passes closer than {PASS_DEBOUNCE_SEC}s.",
        f"Pass min distance: ignore same-team passes shorter than {PASS_MIN_DIST_PX}px.",
        "Renderer now filters overlay circles by MY_TEAM_ID only.",
        "Pass rule: same-team transfer between different players is counted directly (ground or air).",
        "ByteTrack thresholds were tightened to reduce tracker ID explosion.",
    ],
}

# 5) Save files
print("Saving statistics files...")
summary_json_path = os.path.join(stats_dir, "summary.json")
team_stats_csv_path = os.path.join(stats_dir, "team_stats.csv")
player_stats_csv_path = os.path.join(stats_dir, "player_stats.csv")
possession_10min_csv_path = os.path.join(stats_dir, "possession_10min.csv")
events_csv_path = os.path.join(stats_dir, "events.csv")
team_thirds_csv_path = os.path.join(stats_dir, "team_possession_by_third.csv")
thirds_10min_csv_path = os.path.join(stats_dir, "possession_by_third_10min.csv")
my_team_thirds_json_path = os.path.join(stats_dir, "my_team_thirds_summary.json")

with open(summary_json_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

with open(my_team_thirds_json_path, "w", encoding="utf-8") as f:
    json.dump(my_team_thirds, f, ensure_ascii=False, indent=2)

with open(team_stats_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(team_stats_rows[0].keys()) if team_stats_rows else [])
    if team_stats_rows:
        writer.writeheader()
        writer.writerows(team_stats_rows)

with open(team_thirds_csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(team_thirds_rows[0].keys()) if team_thirds_rows else [])
    if team_thirds_rows:
        writer.writeheader()
        writer.writerows(team_thirds_rows)

with open(player_stats_csv_path, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["canonical_id", "team_id", "touches", "possession_seconds", "distance_covered_px"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(player_rows)

with open(possession_10min_csv_path, "w", newline="", encoding="utf-8") as f:
    fieldnames = ["window_index", "start_sec", "end_sec", "team_0_possession_pct", "team_1_possession_pct", "unknown_pct", "frames"]
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(window_rows)

with open(thirds_10min_csv_path, "w", newline="", encoding="utf-8") as f:
    if window_thirds_rows:
        writer = csv.DictWriter(f, fieldnames=list(window_thirds_rows[0].keys()))
        writer.writeheader()
        writer.writerows(window_thirds_rows)

with open(events_csv_path, "w", newline="", encoding="utf-8") as f:
    if len(events) > 0:
        keys = sorted({k for row in events for k in row.keys()})
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(events)
    else:
        writer = csv.writer(f)
        writer.writerow(["info"])
        writer.writerow(["No pass/turnover events were detected with current thresholds."])

print("Stats saved to:", stats_dir)
print("-", summary_json_path)
print("-", my_team_thirds_json_path)
print("-", team_stats_csv_path)
print("-", team_thirds_csv_path)
print("-", player_stats_csv_path)
print("-", possession_10min_csv_path)
print("-", thirds_10min_csv_path)
print("-", events_csv_path)


In [ ]:
# Visual analytics for thirds possession
import os
import json
import subprocess
import sys
import matplotlib.pyplot as plt

try:
    import pandas as pd
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas"], check=False)
    import pandas as pd

stats_dir = os.path.join(REPO_ROOT, "examples", "soccer", "data", "stats")
team_thirds_path = os.path.join(stats_dir, "team_possession_by_third.csv")
thirds_10min_path = os.path.join(stats_dir, "possession_by_third_10min.csv")
my_team_path = os.path.join(stats_dir, "my_team_thirds_summary.json")

for p in [team_thirds_path, thirds_10min_path, my_team_path]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing {p}. Run the stats cell first.")

team_thirds_df = pd.read_csv(team_thirds_path)
thirds_10_df = pd.read_csv(thirds_10min_path)
with open(my_team_path, "r", encoding="utf-8") as f:
    my_team_thirds = json.load(f)

team_thirds_df["team_label"] = team_thirds_df["team_id"].apply(
    lambda t: "My Team" if int(t) == int(MY_TEAM_ID) else "Opponent"
)

print("My team thirds summary:")
display(pd.DataFrame([my_team_thirds]))
print("\nTeam-level thirds table:")
display(team_thirds_df[[
    "team_id", "team_label", "goal_side",
    "defensive_sec", "middle_sec", "offensive_sec",
    "defensive_pct_of_team_possession", "middle_pct_of_team_possession", "offensive_pct_of_team_possession"
]])

fig1, ax1 = plt.subplots(figsize=(10, 5))
plot_df = team_thirds_df.set_index("team_label")[["defensive_sec", "middle_sec", "offensive_sec"]]
plot_df.plot(kind="bar", stacked=True, ax=ax1, color=["#1f77b4", "#ffbf00", "#d62728"] )
ax1.set_title("Possession Time by Third (Seconds)")
ax1.set_ylabel("Seconds")
ax1.set_xlabel("")
ax1.legend(title="Third")
plt.xticks(rotation=0)
plt.tight_layout()

my_10_df = thirds_10_df[thirds_10_df["team_id"] == int(MY_TEAM_ID)].copy()
my_10_df = my_10_df.sort_values(["window_index"])
my_10_df["window_label"] = my_10_df.apply(
    lambda r: f"{int(r['start_sec']//60)}-{int(r['end_sec']//60)} min", axis=1
)

fig2, ax2 = plt.subplots(figsize=(12, 5))
my_10_plot = my_10_df.set_index("window_label")[["defensive_sec", "middle_sec", "offensive_sec"]]
my_10_plot.plot(kind="bar", stacked=True, ax=ax2, color=["#1f77b4", "#ffbf00", "#d62728"] )
ax2.set_title("My Team Possession by Third (Per 10-Minute Window)")
ax2.set_ylabel("Seconds")
ax2.set_xlabel("Window")
ax2.legend(title="Third")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

chart_team_path = os.path.join(stats_dir, "chart_team_thirds_seconds.png")
chart_my10_path = os.path.join(stats_dir, "chart_my_team_thirds_10min.png")
fig1.savefig(chart_team_path, dpi=150, bbox_inches="tight")
fig2.savefig(chart_my10_path, dpi=150, bbox_inches="tight")
plt.show()

print("Saved charts:")
print("-", chart_team_path)
print("-", chart_my10_path)

In [ ]:
# Download all stats as one ZIP (Colab) or print path (RunPod/local)
import os
import shutil

stats_zip_path = os.path.join(stats_dir, "stats_bundle.zip")
if os.path.exists(stats_zip_path):
    os.remove(stats_zip_path)

shutil.make_archive(stats_zip_path.replace('.zip', ''), 'zip', stats_dir)
print("Stats bundle:", stats_zip_path)

try:
    from google.colab import files as colab_files  # type: ignore
    colab_files.download(stats_zip_path)
except Exception:
    print("Not running in Colab; download manually from the path above.")

In [ ]:
# Preview result in notebook
from IPython.display import HTML
from base64 import b64encode

mp4 = open(TARGET_VIDEO_PATH, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width="900" controls><source src="{data_url}" type="video/mp4"></video>')

In [ ]:
# Download result file (Colab) or print path (RunPod/local)
import os

if not os.path.exists(TARGET_VIDEO_PATH):
    raise FileNotFoundError(f"Result video not found: {TARGET_VIDEO_PATH}")

print("Result video:", TARGET_VIDEO_PATH)
try:
    from google.colab import files as colab_files  # type: ignore
    colab_files.download(TARGET_VIDEO_PATH)
except Exception:
    print("Not running in Colab; download manually from the path above.")